In [ ]:
from fusiontimeseries.loralib.layers import BilinearLoRA, Linear, RSSBilinearLoRA
import torch
from torch import nn
from fusiontimeseries.experiments.config import FinetuningConfig
from fusiontimeseries.experiments.dataset import FluxDataset
from fusiontimeseries.experiments.model import get_model

from fusiontimeseries.experiments.model import evaluate
import json


from pathlib import Path

In [ ]:
def parse_adapter_type(
    folder_name: str,
) -> type[Linear | BilinearLoRA | RSSBilinearLoRA]:
    """Parse the adapter type from the folder name."""
    if "BilinearLoRA" in folder_name and "RSS" not in folder_name:
        return BilinearLoRA
    elif "RSSBilinearLoRA" in folder_name:
        return RSSBilinearLoRA
    elif "Linear" in folder_name:
        return Linear
    else:
        raise ValueError(
            f"Could not determine adapter type from folder name: {folder_name}"
        )


def load_trained_model(
    output_dir: Path,
    config: FinetuningConfig,
    Adapter: type[Linear | BilinearLoRA | RSSBilinearLoRA],
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> nn.Module:
    """Load a trained model from an output directory."""

    # Initialize the model with the same architecture
    model = get_model(
        config=config,
        output_dir=output_dir,  # Not used for loading, but required by signature
        device=device,
        Adapter=Adapter,
    )

    # Load the saved LoRA weights
    weights_path = output_dir / "lora_weights.pt"
    if not weights_path.exists():
        raise FileNotFoundError(f"Weights file not found: {weights_path}")

    lora_weights = torch.load(weights_path, map_location=device)
    model.load_state_dict(lora_weights, strict=False)

    # Ensure the entire model is on the correct device
    model = model.to(device)

    return model.eval()

In [ ]:
outputs_dir = Path(".").absolute().parent.parent.parent / "outputs"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Get all output folders except those starting with "TEST-"
output_folders = [
    folder
    for folder in outputs_dir.iterdir()
    if folder.is_dir() and not folder.name.startswith("TEST-")
]

print(f"Found {len(output_folders)} output folders to evaluate")

In [ ]:
output_folders

In [ ]:
output_folder = output_folders[5]

folder_name = output_folder.name
print("\n" + "=" * 80)
print(f"Evaluating: {folder_name}")
print("=" * 80)

# Load the config
config_path = output_folder / "fts_config.json"
if not config_path.exists():
    print(f"  ⚠️  Config not found, skipping: {config_path}")


with open(config_path, "r") as f:
    config_dict = json.load(f)
config = FinetuningConfig(**config_dict)

# Parse adapter type from folder name
Adapter = parse_adapter_type(folder_name)
print(f"  Adapter type: {Adapter.__name__}")
print(
    f"  Config: eval_cutoff={config.eval_context_cutoff}, "
    f"context_len={config.context_length}, subsampling={config.subsampling}"
)

# Load the trained model
print("  Loading model...")
model = load_trained_model(
    output_dir=output_folder,
    config=config,
    Adapter=Adapter,
    device=device,
)

In [ ]:
train_config = (
    {
        "run_folder_prefix": "DataScaling-b1-",
        "eval_context_cutoff": 1,
        "train_context_cutoffs": [1, 129, 257, 385, 513, 641, 672],
        "subsampling": False,
        "context_length": 800,
        "adapter": RSSBilinearLoRA,
        "train_namespaces": [
            "gyroswin_train",
            "batch_1",
        ],
        "val_namespaces": ["batch_6"],
        "test_namespaces": ["batch_9"],
    },
)

In [ ]:
test_dataset = FluxDataset(namespaces=["batch_9"], config=config)

In [ ]:
test_results = evaluate(
    model=model,
    config=config,
    data=test_dataset.flux_data,
    device=device,
)

In [ ]:
results_file = output_folder / "batch9_test_results.json"
with open(results_file, "w") as f:
    json.dump(test_results, f, indent=4)
print(f"  Results saved to: {results_file}")